Map images
Load the image manifest and the product/building reference tables.

In [2]:
from datetime import date, datetime
from numbers import Integral
from pathlib import Path

from openpyxl import load_workbook
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    BooleanType,
    DateType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

In [3]:
import os
import sys
import pyspark

from pathlib import Path
from pyspark.sql import SparkSession

# Environment
pyspark_home = Path(pyspark.__file__).parent

os.environ["SPARK_HOME"] = str(pyspark_home)
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Spark
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("SimCompanies Gold Layer")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Python:", sys.executable)
print("Context stopped:", spark.sparkContext._jsc is None)

Spark: 3.5.1
Python: c:\Users\Prezes\AppData\Local\Programs\Python\Python310\python.exe
Context stopped: False


In [4]:
# Find the project root from the current notebook working directory.
# This works when the notebook is run from the project root or a subfolder.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "data" / "source" / "Book1.xlsx").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Expected data/source/Book1.xlsx "
        "in the current directory or one of its parent directories."
    )

MANIFEST_PATH = PROJECT_ROOT / "data" / "images" / "image_manifest.csv"
BUILDINGS_DIR = PROJECT_ROOT / "data" / "images" / "buildings"
PRODUCTS_DIR = PROJECT_ROOT / "data" / "images" / "products"
EXCEL_PATH = PROJECT_ROOT / "data" / "source" / "Book1.xlsx"

print(f"Project root: {PROJECT_ROOT}")
print(f"Manifest:     {MANIFEST_PATH}")
print(f"Buildings:    {BUILDINGS_DIR}")
print(f"Products:     {PRODUCTS_DIR}")
print(f"Excel:        {EXCEL_PATH}")

Project root: c:\Projekty\SimCompanies
Manifest:     c:\Projekty\SimCompanies\data\images\image_manifest.csv
Buildings:    c:\Projekty\SimCompanies\data\images\buildings
Products:     c:\Projekty\SimCompanies\data\images\products
Excel:        c:\Projekty\SimCompanies\data\source\Book1.xlsx


In [4]:
spark = (
    SparkSession.builder
    .appName("SimCompaniesImageMapping")
    .getOrCreate()
)

In [5]:
df_image_manifest = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(str(MANIFEST_PATH))
)

df_image_manifest.show(5, truncate=False)

+--------+----------------------------+------------------------------------------+-------------------------------------------------------------------------------------------------------------+
|kind    |name                        |file                                      |source_url                                                                                                   |
+--------+----------------------------+------------------------------------------+-------------------------------------------------------------------------------------------------------------+
|building|academy_tier01              |buildings/academy_tier01.png              |https://www.simcompanies.com/static/images/buildings/other/academy_tier01.24c352b54d3e.png                   |
|building|bank_tier01                 |buildings/bank_tier01.png                 |https://www.simcompanies.com/static/images/buildings/other/bank_tier01.3a671cf2ef85.png                      |
|building|aerospace_electronics_tie

In [6]:
def _infer_spark_type(values):
    non_null_values = [value for value in values if value is not None]

    if not non_null_values:
        return StringType()
    if all(isinstance(value, bool) for value in non_null_values):
        return BooleanType()
    if all(
        isinstance(value, Integral) and not isinstance(value, bool)
        for value in non_null_values
    ):
        return LongType()
    if all(
        isinstance(value, (int, float)) and not isinstance(value, bool)
        for value in non_null_values
    ):
        return DoubleType()
    if all(isinstance(value, datetime) for value in non_null_values):
        return TimestampType()
    if all(isinstance(value, date) for value in non_null_values):
        return DateType()
    return StringType()


def read_excel_sheet_as_spark_df(excel_path, sheet_name):
    workbook = load_workbook(
        filename=excel_path,
        read_only=True,
        data_only=True,
    )

    try:
        if sheet_name not in workbook.sheetnames:
            raise ValueError(
                f"Sheet {sheet_name!r} not found in {excel_path}. "
                f"Available sheets: {workbook.sheetnames}"
            )

        worksheet = workbook[sheet_name]
        rows = worksheet.iter_rows(values_only=True)
        header = next(rows, None)

        if header is None:
            raise ValueError(f"Sheet {sheet_name!r} is empty.")

        # Ignore trailing empty header cells.
        header = list(header)
        while header and header[-1] is None:
            header.pop()

        if not header:
            raise ValueError(f"Sheet {sheet_name!r} has no column headers.")

        columns = [
            str(value).strip() if value is not None and str(value).strip() else f"_c{index}"
            for index, value in enumerate(header)
        ]

        data = []
        for row in rows:
            values = tuple(row[:len(columns)])
            values += (None,) * (len(columns) - len(values))
            if any(value is not None for value in values):
                data.append(values)
    finally:
        workbook.close()

    schema_fields = []
    for index, column_name in enumerate(columns):
        column_values = [row[index] for row in data]
        spark_type = _infer_spark_type(column_values)
        schema_fields.append(StructField(column_name, spark_type, nullable=True))

    schema = StructType(schema_fields)
    normalized_data = []
    for row in data:
        normalized_row = []
        for value, field in zip(row, schema.fields):
            if value is None:
                normalized_row.append(None)
            elif isinstance(field.dataType, StringType):
                normalized_row.append(str(value))
            elif isinstance(field.dataType, LongType):
                normalized_row.append(int(value))
            elif isinstance(field.dataType, DoubleType):
                normalized_row.append(float(value))
            else:
                normalized_row.append(value)
        normalized_data.append(tuple(normalized_row))

    return spark.createDataFrame(normalized_data, schema=schema)

In [7]:
df_products = read_excel_sheet_as_spark_df(EXCEL_PATH, "products")
df_buildings = read_excel_sheet_as_spark_df(EXCEL_PATH, "buildings")

print(f"Products:  {df_products.count()} rows, {len(df_products.columns)} columns")
print(f"Buildings: {df_buildings.count()} rows, {len(df_buildings.columns)} columns")

Products:  151 rows, 3 columns
Buildings: 45 rows, 3 columns


In [8]:
df_products.show(5, truncate=False)
df_buildings.show(5, truncate=False)

+----------+-------+-----------+
|product_id|name   |category   |
+----------+-------+-----------+
|1         |Seeds  |Agriculture|
|2         |Apples |Agriculture|
|3         |Oranges|Agriculture|
|4         |Grapes |Agriculture|
|5         |Grain  |Agriculture|
+----------+-------+-----------+
only showing top 5 rows

+-----------+--------------+----+
|building_id|name          |wage|
+-----------+--------------+----+
|1          |Slaughterhouse|414 |
|2          |Mill          |380 |
|3          |Ranch         |138 |
|4          |Grocery store |138 |
|5          |Farm          |104 |
+-----------+--------------+----+
only showing top 5 rows



In [9]:
# Register Spark DataFrames as temporary SQL views

df_image_manifest.createOrReplaceTempView("df_image_manifest")
df_products.createOrReplaceTempView("df_products")
df_buildings.createOrReplaceTempView("df_buildings")

## Products

In [25]:
results = spark.sql("""
    SELECT
        COUNT(*) AS matching_rows
    FROM df_image_manifest m
    INNER JOIN df_products p
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
    WHERE m.kind = 'product'
""").show()

+-------------+
|matching_rows|
+-------------+
|           78|
+-------------+



In [22]:
unmatched_files = spark.sql("""
    SELECT
        m.name AS image_name,
        m.file,
        m.source_url
    FROM df_image_manifest m
    LEFT ANTI JOIN df_products p
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
    WHERE m.kind = 'product'
    ORDER BY m.name
""")

display(unmatched_files.toPandas())

,image_name,file,source_url
0,aero-research,products/aero-research.png,https://www.simcompanies.com/static/images/res...
1,apple-cider,products/apple-cider.png,https://www.simcompanies.com/static/images/res...
2,apple-pie,products/apple-pie.png,https://www.simcompanies.com/static/images/res...
3,attitude-control,products/attitude-control.png,https://www.simcompanies.com/static/images/res...
4,automotive-research,products/automotive-research.png,https://www.simcompanies.com/static/images/res...
...,...,...,...
68,sub-orbital-second-stage,products/sub-orbital-second-stage.png,https://www.simcompanies.com/static/images/res...
69,vegetable-oil,products/vegetable-oil.png,https://www.simcompanies.com/static/images/res...
70,witch-costume,products/witch-costume.png,https://www.simcompanies.com/static/images/res...
71,xmas-crackers,products/xmas-crackers.png,https://www.simcompanies.com/static/images/res...


In [23]:
unmatched_products = spark.sql("""
    SELECT
        p.product_id,
        p.name AS product_name,
        p.category
    FROM df_products p
    LEFT ANTI JOIN df_image_manifest m
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
        AND m.kind = 'product'
    ORDER BY p.name
""")

display(unmatched_products.toPandas())

,product_id,product_name,category
0,140,Aerospace research,Research
1,20,Apple Cider,Food
2,150,Apple icecream,Seasonal
3,28,Apple pie,Food
4,97,Attitude control,Aerospace
...,...,...,...
68,105,Sub-orbital rocket,Aerospace
69,37,Vegetable oil,Food
70,142,Witch costume,Seasonal
71,145,Xmas crackers,Seasonal


In [26]:
from pyspark.sql import functions as F

df_image_manifest.createOrReplaceTempView("df_image_manifest_original")

df_image_manifest_normalized = df_image_manifest.withColumn(
    "name",
    F.regexp_replace(F.col("name"), "-", " ")
)

df_image_manifest_normalized.createOrReplaceTempView("df_image_manifest")

In [27]:
spark.sql("""
    SELECT COUNT(*) AS matching_rows
    FROM df_image_manifest m
    INNER JOIN df_products p
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
    WHERE m.kind = 'product'
""").show()

+-------------+
|matching_rows|
+-------------+
|          122|
+-------------+



In [28]:
unmatched_files = spark.sql("""
    SELECT
        m.name AS image_name,
        m.file,
        m.source_url
    FROM df_image_manifest m
    LEFT ANTI JOIN df_products p
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
    WHERE m.kind = 'product'
    ORDER BY m.name
""")

display(unmatched_files.toPandas())

,image_name,file,source_url
0,aero research,products/aero-research.png,https://www.simcompanies.com/static/images/res...
1,car interior,products/car-interior.png,https://www.simcompanies.com/static/images/res...
2,carbon fiber,products/carbon-fiber.png,https://www.simcompanies.com/static/images/res...
3,cocoa beans,products/cocoa-beans.png,https://www.simcompanies.com/static/images/res...
4,coffee ground,products/coffee-ground.png,https://www.simcompanies.com/static/images/res...
5,cow,products/cow.png,https://www.simcompanies.com/static/images/res...
6,economy e car,products/economy-e-car.png,https://www.simcompanies.com/static/images/res...
7,fuel tank,products/fuel-tank.png,https://www.simcompanies.com/static/images/res...
8,gold watch,products/gold-watch.png,https://www.simcompanies.com/static/images/res...
9,gravy boat,products/gravy-boat.png,https://www.simcompanies.com/static/images/res...


In [29]:
unmatched_products = spark.sql("""
    SELECT
        p.product_id,
        p.name AS product_name,
        p.category
    FROM df_products p
    LEFT ANTI JOIN df_image_manifest m
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
        AND m.kind = 'product'
    ORDER BY p.name
""")

display(unmatched_products.toPandas())

,product_id,product_name,category
0,140,Aerospace research,Research
1,150,Apple icecream,Seasonal
2,85,Basic interior,Automotive
3,49,Beams,Construction
4,127,Carbon fibers,Resources
5,149,Chocolate icecream,Seasonal
6,13,Cocoa,Agriculture
7,17,Coffee powder,Food
8,6,Cows,Agriculture
9,88,Economy e-car,Automotive


In [30]:
image_name_mapping = {
    "aero research": "Aerospace research",
    "car interior": "Basic interior",
    "carbon fiber": "Carbon fibers",
    "cocoa beans": "Cocoa",
    "coffee ground": "Coffee powder",
    "cow": "Cows",
    "economy e car": "Economy e-car",
    "fuel tank": "Propellant tank",
    "gold watch": "Luxury watch",
    "gravy boat": "Sauce",
    "high grade e components": "High grade e-comps",
    "icecream apple": "Apple icecream",
    "icecream chocolate": "Chocolate icecream",
    "jack o lantern": "Jack o'lantern",
    "jumbojet2": "Jumbo jet",
    "luxury e car": "Luxury e-car",
    "meatballs": "Meat balls",
    "on board computer": "On-board computer",
    "pig": "Pigs",
    "pizza": "Frozen pizza",
    "private jet": "Luxury jet",
    "samosas": "Samosa",
    "simmi shoes": "Stiletto heel",
    "single engine": "Single engine plane",
    "smart phones": "Smartphones",
    "solid rocket": "Solid fuel booster",
    "steel beams": "Beams",
    "sub orbital rocket2": "Sub-orbital rocket",
    "sub orbital second stage": "Sub-orbital 2nd stage"
}

In [34]:
from pyspark.sql import functions as F

mapping_expr = F.create_map(
    *[
        item
        for key, value in image_name_mapping.items()
        for item in (F.lit(key), F.lit(value))
    ]
)

df_image_manifest_mapped = (
    df_image_manifest_normalized
    .withColumn(
        "name",
        F.coalesce(
            F.element_at(mapping_expr, F.col("name")),
            F.concat(
                F.upper(F.substring(F.col("name"), 1, 1)),
                F.lower(F.substring(F.col("name"), 2, 1000))
            )
        )
    )
)

df_image_manifest_mapped.createOrReplaceTempView("df_image_manifest_mapped")

In [35]:
spark.sql("""
    SELECT COUNT(*) AS matching_rows
    FROM df_image_manifest_mapped m
    INNER JOIN df_products p
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
    WHERE m.kind = 'product'
""").show()

+-------------+
|matching_rows|
+-------------+
|          151|
+-------------+



In [45]:
spark.sql("""
    SELECT *
    FROM df_image_manifest_mapped m
    INNER JOIN df_products p
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
    WHERE m.kind = 'product'
""").show()

+-------+-------------+--------------------+--------------------+----------+-------------+-----------+
|   kind|         name|                file|          source_url|product_id|         name|   category|
+-------+-------------+--------------------+--------------------+----------+-------------+-----------+
|product|        Seeds|  products/seeds.png|https://www.simco...|         1|        Seeds|Agriculture|
|product|       Apples| products/apples.png|https://www.simco...|         2|       Apples|Agriculture|
|product|      Oranges|products/oranges.png|https://www.simco...|         3|      Oranges|Agriculture|
|product|       Grapes| products/grapes.png|https://www.simco...|         4|       Grapes|Agriculture|
|product|        Grain|  products/grain.png|https://www.simco...|         5|        Grain|Agriculture|
|product|         Cows|    products/cow.png|https://www.simco...|         6|         Cows|Agriculture|
|product|         Pigs|    products/pig.png|https://www.simco...|        

In [44]:
unmatched_files = spark.sql("""
    SELECT
        m.name AS image_name,
        m.file,
        m.source_url
    FROM df_image_manifest_mapped m
    LEFT ANTI JOIN df_products p
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(p.name))
    WHERE m.kind = 'product'
    ORDER BY m.name
""")

display(unmatched_files.toPandas())

,image_name,file,source_url


## Save products

In [50]:
BUILDINGS_DIR = PROJECT_ROOT / "data" / "images" / "buildings"
PRODUCTS_DIR = PROJECT_ROOT / "data" / "images" / "products"

In [51]:
print(PRODUCTS_DIR)
print(PRODUCTS_DIR.is_absolute())
print((PRODUCTS_DIR / "seeds.png").exists())

c:\Projekty\SimCompanies\data\images\products
True
True


In [52]:
row = (
    df_image_manifest_mapped
    .filter(F.col("kind") == "product")
    .select("file", "name")
    .first()
)

old_path = PRODUCTS_DIR / Path(row["file"]).name

print("Manifest:", row["file"])
print("Resolved:", old_path)
print("Exists:", old_path.exists())

Manifest: products/BFR.png
Resolved: c:\Projekty\SimCompanies\data\images\products\BFR.png
Exists: True


In [54]:
product_images = (
    df_image_manifest_mapped
    .filter(F.col("kind") == "product")
    .select("file", "name")
    .collect()
)

renamed = 0
skipped = 0

for row in product_images:
    old_path = PRODUCTS_DIR / Path(row["file"]).name

    final_name = row["name"].strip()
    final_name = final_name[:1].upper() + final_name[1:].lower()
    new_path = PRODUCTS_DIR / f"{final_name}.png"

    if not old_path.exists():
        print(f"Missing source: {old_path.name}")
        skipped += 1
        continue

    # Exact name match (case-sensitive)
    if old_path.name == new_path.name:
        skipped += 1
        continue

    # Case-only rename on Windows
    if old_path.name.lower() == new_path.name.lower():
        temp_path = old_path.with_name(f"__temp_{old_path.name}")

        old_path.rename(temp_path)
        temp_path.rename(new_path)

    elif new_path.exists():
        print(f"Target already exists: {new_path.name}")
        skipped += 1
        continue

    else:
        old_path.rename(new_path)

    print(f"{old_path.name} -> {new_path.name}")
    renamed += 1

print(f"\nRenamed: {renamed}")
print(f"Skipped: {skipped}")

BFR.png -> Bfr.png
Missing source: aero-research.png
aluminium.png -> Aluminium.png
Missing source: apple-cider.png
Missing source: apple-pie.png
apples.png -> Apples.png
Missing source: attitude-control.png
Missing source: automotive-research.png
batteries.png -> Batteries.png
bauxite.png -> Bauxite.png
bread.png -> Bread.png
Missing source: breeding-research.png
bricks.png -> Bricks.png
bulldozer.png -> Bulldozer.png
butter.png -> Butter.png
Missing source: car-body.png
Missing source: car-interior.png
Missing source: carbon-composite.png
Missing source: carbon-fiber.png
cement.png -> Cement.png
cheese.png -> Cheese.png
chemicals.png -> Chemicals.png
Missing source: chemistry-research.png
chocolate.png -> Chocolate.png
clay.png -> Clay.png
cockpit.png -> Cockpit.png
cocktails.png -> Cocktails.png
Missing source: cocoa-beans.png
Missing source: coffee-beans.png
Missing source: coffee-ground.png
Missing source: combustion-engine.png
Missing source: construction-units.png
cotton.png -> 

## Buildings

In [55]:
spark.sql("""
    SELECT COUNT(*) AS matching_rows
    FROM df_image_manifest m
    INNER JOIN df_buildings b
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(b.name))
    WHERE m.kind = 'building'
""").show()

+-------------+
|matching_rows|
+-------------+
|            0|
+-------------+



In [61]:
unmatched_buildings = spark.sql("""
    SELECT
        m.name AS image_name,
        m.file,
        m.source_url
    FROM df_image_manifest m
    LEFT ANTI JOIN df_buildings b
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(b.name))
    WHERE m.kind = 'building'
    ORDER BY m.name
    LIMIT 5
""")

display(unmatched_buildings.toPandas())

,image_name,file,source_url
0,academy_tier01,buildings/academy_tier01.png,https://www.simcompanies.com/static/images/bui...
1,aerospace_electronics_tier01,buildings/aerospace_electronics_tier01.png,https://www.simcompanies.com/static/images/bui...
2,aerospace_factory_tier01,buildings/aerospace_factory_tier01.png,https://www.simcompanies.com/static/images/bui...
3,automotive_research_tier01,buildings/automotive_research_tier01.png,https://www.simcompanies.com/static/images/bui...
4,bakery_tier01,buildings/bakery_tier01.png,https://www.simcompanies.com/static/images/bui...


In [62]:
missing_building_images = spark.sql("""
    SELECT
        b.building_id,
        b.name
    FROM df_buildings b
    LEFT ANTI JOIN df_image_manifest m
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(b.name))
        AND m.kind = 'building'
    ORDER BY b.name
    LIMIT 5
""")

display(missing_building_images.toPandas())

,building_id,name
0,24,Aerospace electronics
1,28,Aerospace factory
2,43,Automotive R&D
3,8,Bakery
4,7,Beverage factory


In [66]:
from pyspark.sql import functions as F

df_image_manifest_normalized = (
    df_image_manifest
    .withColumn(
        "name",
        F.when(
            F.col("kind") == "building",
            F.regexp_replace(F.col("name"), "_tier01$", "")
        ).otherwise(F.col("name"))
    )
    .withColumn(
        "name",
        F.when(
            F.col("kind") == "building",
            F.regexp_replace(F.col("name"), "_", " ")
        ).otherwise(F.col("name"))
    )
    .withColumn(
        "name",
        F.when(
            F.col("kind") == "building",
            F.concat(
                F.upper(F.substring(F.trim(F.col("name")), 1, 1)),
                F.lower(F.substring(F.trim(F.col("name")), 2, 1000))
            )
        ).otherwise(F.col("name"))
    )
)

df_image_manifest_normalized.createOrReplaceTempView(
    "df_image_manifest_normalized"
)

In [67]:
spark.sql("""
    SELECT
        m.name AS image_name,
        m.file
    FROM df_image_manifest_normalized m
    WHERE m.kind = 'building'
    LIMIT 10
""").show(truncate=False)

+---------------------+------------------------------------------+
|image_name           |file                                      |
+---------------------+------------------------------------------+
|Academy              |buildings/academy_tier01.png              |
|Bank                 |buildings/bank_tier01.png                 |
|Aerospace electronics|buildings/aerospace_electronics_tier01.png|
|Aerospace factory    |buildings/aerospace_factory_tier01.png    |
|Bakery               |buildings/bakery_tier01.png               |
|Beverage factory     |buildings/beverage_factory_tier01.png     |
|Car factory          |buildings/car_factory_tier01.png          |
|Catering             |buildings/catering_tier01.png             |
|Concrete plant       |buildings/concrete_plant_tier01.png       |
|Construction factory |buildings/construction_factory_tier01.png |
+---------------------+------------------------------------------+



In [68]:
spark.sql("""
    SELECT COUNT(*) AS matching_rows
    FROM df_image_manifest m
    INNER JOIN df_buildings b
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(b.name))
    WHERE m.kind = 'building'
""").show()

+-------------+
|matching_rows|
+-------------+
|            0|
+-------------+



In [70]:
df_image_manifest_normalized.createOrReplaceTempView(
    "df_image_manifest_normalized"
)

In [71]:
unmatched_buildings = spark.sql("""
    SELECT
        m.name AS image_name,
        m.file,
        m.source_url
    FROM df_image_manifest_normalized m
    LEFT ANTI JOIN df_buildings b
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(b.name))
    WHERE m.kind = 'building'
    ORDER BY m.name
    LIMIT 5
""")

display(unmatched_buildings.toPandas())

,image_name,file,source_url
0,Academy,buildings/academy_tier01.png,https://www.simcompanies.com/static/images/bui...
1,Automotive research,buildings/automotive_research_tier01.png,https://www.simcompanies.com/static/images/bui...
2,Bank,buildings/bank_tier01.png,https://www.simcompanies.com/static/images/bui...
3,Breeding laboratory,buildings/breeding_laboratory_tier01.png,https://www.simcompanies.com/static/images/bui...
4,Chemistry laboratory,buildings/chemistry_laboratory_tier01.png,https://www.simcompanies.com/static/images/bui...


In [72]:
spark.sql("""
    SELECT COUNT(*) AS matching_rows
    FROM df_image_manifest_normalized m
    INNER JOIN df_buildings b
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(b.name))
    WHERE m.kind = 'building'
""").show()

+-------------+
|matching_rows|
+-------------+
|           36|
+-------------+



In [74]:
unmatched_buildings = spark.sql("""
    SELECT
        m.name AS image_name,
        b.name AS building_name
    FROM df_image_manifest_normalized m
    FULL OUTER JOIN df_buildings b
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(b.name))
    WHERE
        m.kind = 'building'
        AND b.name IS NULL
        OR m.name IS NULL
    ORDER BY COALESCE(m.name, b.name)
""")

display(unmatched_buildings.toPandas())

,image_name,building_name
0,Academy,None
1,None,Automotive R&D
2,Automotive research,None
3,Bank,None
4,None,Breeding labolatory
5,Breeding laboratory,None
6,None,Chemistry labolatory
7,Chemistry laboratory,None
8,None,Fashion & design
9,Fashion research,None


In [75]:
building_name_mapping = {
    "Automotive research": "Automotive R&D",
    "Breeding laboratory": "Breeding labolatory",
    "Chemistry laboratory": "Chemistry labolatory",
    "Fashion research": "Fashion & design",
    "Forrest nursery tier01 back": "Forest nursery",
    "Forrest nursery tier01 front": "Forest nursery",
    "Grocery store idle": "Grocery store",
    "Physics laboratory": "Physics labolatory",
    "Restaurant tier01-low": "Restaurant",
    "Software research": "Software R&D",
}

In [76]:
mapping_expr = F.create_map(
    *[
        item
        for key, value in building_name_mapping.items()
        for item in (F.lit(key), F.lit(value))
    ]
)

df_image_manifest_mapped = (
    df_image_manifest_normalized
    .withColumn(
        "name",
        F.when(
            F.col("kind") == "building",
            F.coalesce(
                F.element_at(mapping_expr, F.col("name")),
                F.col("name")
            )
        ).otherwise(F.col("name"))
    )
)

df_image_manifest_mapped.createOrReplaceTempView(
    "df_image_manifest_mapped"
)

In [77]:
unmatched_buildings = spark.sql("""
    SELECT
        m.name AS image_name,
        b.name AS building_name
    FROM (
        SELECT *
        FROM df_image_manifest_mapped
        WHERE kind = 'building'
    ) m
    FULL OUTER JOIN df_buildings b
        ON TRIM(LOWER(m.name)) = TRIM(LOWER(b.name))
    WHERE m.name IS NULL OR b.name IS NULL
    ORDER BY COALESCE(m.name, b.name)
""")

display(unmatched_buildings.toPandas())

,image_name,building_name
0,Academy,None
1,Bank,None
2,Halloween market,None
3,Lake,None
4,Park,None
5,Pool market,None
6,Spring market,None
7,Temple,None


## Save

In [78]:
building_images = (
    df_image_manifest_mapped
    .filter(F.col("kind") == "building")
    .select("file", "name")
    .collect()
)

renamed = 0
skipped = 0

for row in building_images:
    old_path = BUILDINGS_DIR / Path(row["file"]).name

    final_name = row["name"].strip()
    final_name = final_name[:1].upper() + final_name[1:].lower()

    # Preserve front/back variants to avoid filename collisions
    source_stem = Path(row["file"]).stem.lower()

    if source_stem.endswith("_back"):
        final_name += " back"
    elif source_stem.endswith("_front"):
        final_name += " front"

    new_path = BUILDINGS_DIR / f"{final_name}.png"

    if not old_path.exists():
        print(f"Missing source: {old_path.name}")
        skipped += 1
        continue

    if old_path.name == new_path.name:
        skipped += 1
        continue

    if old_path.name.lower() == new_path.name.lower():
        temp_path = old_path.with_name(f"__temp_{old_path.name}")
        old_path.rename(temp_path)
        temp_path.rename(new_path)

    elif new_path.exists():
        print(f"Target already exists: {new_path.name}")
        skipped += 1
        continue

    else:
        old_path.rename(new_path)

    print(f"{old_path.name} -> {new_path.name}")
    renamed += 1

print(f"\nRenamed: {renamed}")
print(f"Skipped: {skipped}")

academy_tier01.png -> Academy.png
bank_tier01.png -> Bank.png
aerospace_electronics_tier01.png -> Aerospace electronics.png
aerospace_factory_tier01.png -> Aerospace factory.png
bakery_tier01.png -> Bakery.png
beverage_factory_tier01.png -> Beverage factory.png
car_factory_tier01.png -> Car factory.png
catering_tier01.png -> Catering.png
concrete_plant_tier01.png -> Concrete plant.png
construction_factory_tier01.png -> Construction factory.png
electronics_factory_tier01.png -> Electronics factory.png
factory_tier01.png -> Factory.png
farm_tier01.png -> Farm.png
fashion_factory_tier01.png -> Fashion factory.png
food_processing_plant_tier01.png -> Food processing plant.png
forrest_nursery_tier01_back.png -> Forest nursery back.png
forrest_nursery_tier01_front.png -> Forest nursery front.png
general_contractor_tier01.png -> General contractor.png
hangar_tier01.png -> Hangar.png
mill_tier01.png -> Mill.png
mine_tier01.png -> Mine.png
oil_rig_tier01.png -> Oil rig.png
power_plant_tier01.png

### Resize images

In [6]:
from PIL import Image

BUILDINGS_DIR = PROJECT_ROOT / "data" / "images" / "buildings"

for path in BUILDINGS_DIR.glob("*.png"):
    with Image.open(path) as img:
        img = img.convert("RGBA")
        img.thumbnail((256, 256), Image.Resampling.LANCZOS)

        canvas = Image.new("RGBA", (256, 256), (0, 0, 0, 0))

        x = (256 - img.width) // 2
        y = (256 - img.height) // 2

        canvas.paste(img, (x, y), img)
        canvas.save(path, optimize=True)

print("Done!")

Done!


In [3]:
from PIL import Image
from pathlib import Path
import io
import base64

BUILDINGS_DIR = PROJECT_ROOT / "data" / "images" / "buildings"
MAX_BASE64_LENGTH = 30000

for path in BUILDINGS_DIR.glob("*.png"):
    with Image.open(path) as original:
        img = original.convert("RGBA")

        while True:
            buffer = io.BytesIO()

            # Quantization reduces PNG size while preserving transparency.
            optimized = img.quantize(
                colors=128,
                method=Image.Quantize.FASTOCTREE
            )

            optimized.save(buffer, format="PNG", optimize=True)

            encoded_length = len(
                base64.b64encode(buffer.getvalue())
            )

            if encoded_length <= MAX_BASE64_LENGTH:
                path.write_bytes(buffer.getvalue())
                break

            # Reduce dimensions while preserving proportions.
            new_size = (
                max(32, int(img.width * 0.85)),
                max(32, int(img.height * 0.85))
            )

            img = img.resize(new_size, Image.Resampling.LANCZOS)

    print(f"{path.name}: {encoded_length} characters")

print("Done!")

Academy.png: 11472 characters
Aerospace electronics.png: 9396 characters
Aerospace factory.png: 8876 characters
Automotive r&d.png: 10012 characters
Bakery.png: 9668 characters
Bank.png: 8604 characters
Beverage factory.png: 9544 characters
Breeding labolatory.png: 10648 characters
Car dealership.png: 9820 characters
Car factory.png: 11104 characters
Catering.png: 9964 characters
Chemistry labolatory.png: 10372 characters
Concrete plant.png: 10604 characters
Construction factory.png: 11536 characters
Electronics factory.png: 10432 characters
Electronics store.png: 9092 characters
Factory.png: 10244 characters
Farm.png: 11352 characters
Fashion & design.png: 10736 characters
Fashion factory.png: 11576 characters
Fashion store.png: 10180 characters
Food processing plant.png: 10296 characters
Forest nursery back.png: 8616 characters
Forest nursery front.png: 2420 characters
Gas station.png: 9540 characters
General contractor.png: 10516 characters
Grocery store.png: 10404 characters
Hallow

In [5]:
from PIL import Image
from pathlib import Path
import io
import base64

BUILDINGS_DIR = PROJECT_ROOT / "data" / "images" / "products"
MAX_BASE64_LENGTH = 30000

for path in BUILDINGS_DIR.glob("*.png"):
    with Image.open(path) as original:
        img = original.convert("RGBA")

        while True:
            buffer = io.BytesIO()

            # Quantization reduces PNG size while preserving transparency.
            optimized = img.quantize(
                colors=128,
                method=Image.Quantize.FASTOCTREE
            )

            optimized.save(buffer, format="PNG", optimize=True)

            encoded_length = len(
                base64.b64encode(buffer.getvalue())
            )

            if encoded_length <= MAX_BASE64_LENGTH:
                path.write_bytes(buffer.getvalue())
                break

            # Reduce dimensions while preserving proportions.
            new_size = (
                max(32, int(img.width * 0.85)),
                max(32, int(img.height * 0.85))
            )

            img = img.resize(new_size, Image.Resampling.LANCZOS)

    print(f"{path.name}: {encoded_length} characters")

print("Done!")

Aerospace research.png: 11516 characters
Aluminium.png: 7532 characters
Apple cider.png: 6012 characters
Apple icecream.png: 12560 characters
Apple pie.png: 6664 characters
Apples.png: 9880 characters
Attitude control.png: 8556 characters
Automotive research.png: 8560 characters
Basic interior.png: 9900 characters
Batteries.png: 5532 characters
Bauxite.png: 10664 characters
Beams.png: 8268 characters
Bfr.png: 6676 characters
Bread.png: 5008 characters
Breeding research.png: 9916 characters
Bricks.png: 10920 characters
Bulldozer.png: 18788 characters
Butter.png: 4620 characters
Car body.png: 11820 characters
Carbon composite.png: 16708 characters
Carbon fibers.png: 12292 characters
Cement.png: 13112 characters
Cheese.png: 3448 characters
Chemicals.png: 8624 characters
Chemistry research.png: 11416 characters
Chocolate icecream.png: 11272 characters
Chocolate.png: 3192 characters
Clay.png: 16680 characters
Cockpit.png: 11780 characters
Cocktails.png: 5464 characters
Cocoa.png: 8832 chara